# PyTorch


Objectif : etre capable de faire des calculs sur tensors, comprendre l'autograd, et ecrire/entrainer un petit modele avec PyTorch. On va droit au but, tout est execute directement.


Plan :
1. Tensors (creation, shape, operations, dtype/device)
2. Autograd (calcul automatique des gradients, graphe de calcul)
3. Construire un modele (nn.Module, couches, activations)
4. Boucle d'entrainement complete (loss, optimizer)
5. Exercice chrono (a faire seul, 15-20 min)
6. Pour aller plus loin (semaine d'approfondissement)

Chaque section a un exemple qui tourne + une **fiche technique** (parametres, ce que fait vraiment la fonction) + un **schema** quand c'est utile + un mini exercice a faire tout de suite apres, pour pas juste lire mais coder.

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

print(torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device utilise:", device)

## 1. Tensors : la brique de base

Un tensor c'est un tableau multi-dimensionnel, comme un array numpy mais qui peut aller sur GPU et calculer des gradients automatiquement.

**Analogie** : pense a un tableur (Excel). Une ligne = un eleve, une colonne = une note dans une matiere. Un tensor 2D, c'est exactement ca : `shape = (nb_eleves, nb_matieres)`. Un tensor 3D, c'est un classeur avec plusieurs feuilles (par exemple une feuille par trimestre).

Points essentiels a retenir :
- `torch.tensor(...)` cree un tensor a partir de donnees Python
- `.shape` donne les dimensions
- `.dtype` donne le type (float32 par defaut pour le deep learning)
- les operations sont vectorisees (pas de boucle for)

In [ ]:
# creation de tensors
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([[1.0, 2.0], [3.0, 4.0]])

print("a:", a, "shape:", a.shape)
print("b:", b, "shape:", b.shape)

# tensors utilitaires
zeros = torch.zeros(2, 3)
ones = torch.ones(2, 3)
rand = torch.rand(2, 3)       # uniforme [0,1)
randn = torch.randn(2, 3)     # normale (0,1)

print(zeros)
print(rand)

### Fiche technique : dtype et device

Deux parametres invisibles mais essentiels de tout tensor.

| Concept | Ce que c'est | Pourquoi ca compte |
|---|---|---|
| `dtype` | le type des nombres stockes (`torch.float32`, `torch.int64`, `torch.bool`...) | l'autograd n'accepte que les floats. Un tensor d'entiers ne peut pas `requires_grad=True` |
| `device` | ou vivent les donnees en memoire (`"cpu"` ou `"cuda"`) | deux tensors sur des devices differents ne peuvent pas etre combines directement -> erreur `RuntimeError` |

**Analogie** : `dtype`, c'est l'unite de mesure (litres vs millilitres) ; `device`, c'est l'entrepot ou est stocke le produit. Deux entrepots differents = il faut d'abord tout rapatrier au meme endroit avant de faire l'inventaire ensemble.

Fonctions courantes :
- `x.to(device)` : deplace (et/ou change le dtype de) `x`, retourne un **nouveau** tensor
- `x.float()`, `x.long()`, `x.bool()` : raccourcis de conversion de dtype
- `x.item()` : sort un tensor a un seul element sous forme de nombre Python (utile pour afficher une loss)
- `x.cpu()`, `x.cuda()` : raccourcis de deplacement

In [ ]:
# dtype et device en pratique
entiers = torch.tensor([1, 2, 3])
print("dtype par defaut pour des ints:", entiers.dtype)

flottants = entiers.float()
print("apres .float():", flottants.dtype)

x = torch.randn(3).to(device)
print("x est sur:", x.device)

# .item() pour sortir un scalaire Python (tres utilise pour print une loss)
valeur = torch.tensor(3.14)
print(type(valeur), "->", type(valeur.item()))

In [ ]:
# operations de base : tout est vectorise, pas de boucle
x = torch.tensor([1.0, 2.0, 3.0])
y = torch.tensor([10.0, 20.0, 30.0])

print(x + y)
print(x * y)          # multiplication element par element
print(x @ y)          # produit scalaire (dot product)

# matrices
M = torch.randn(3, 4)
N = torch.randn(4, 2)
print((M @ N).shape)   # multiplication matricielle -> (3, 2)

# reshape : tres utilise pour preparer des batchs
t = torch.arange(12)          # [0, 1, ..., 11]
print(t.reshape(3, 4))
print(t.reshape(3, 4).view(-1))   # remettre a plat

### Fiche technique : `*` vs `@`

C'est LA confusion numero 1 des debutants.

- `x * y` : multiplication **element par element**. Exige que les deux tensors aient la meme shape (ou soient broadcastables). Resultat : meme shape que l'entree.
- `x @ y` (equivalent a `torch.matmul(x, y)`) : produit **matriciel**. Exige que la derniere dimension de `x` egale l'avant-derniere dimension de `y`. `(3,4) @ (4,2) -> (3,2)`.

**Analogie** : `*`, c'est comme comparer deux paniers de courses article par article (prix unitaire fois quantite, pour chaque produit). `@`, c'est calculer une facture totale a partir de quantites et de prix qui ne sont pas alignes de la meme facon chaque ligne du resultat combine *toute* une ligne de `x` avec *toute* une colonne de `y`.

`reshape` vs `view` : les deux changent la forme sans changer les donnees. `view` exige que le tensor soit "contigu" en memoire (sinon erreur), `reshape` fait automatiquement une copie si besoin. En debut de carriere : utilise `reshape` par defaut, c'est plus tolerant.

In [ ]:
# broadcasting : PyTorch etend automatiquement les dimensions compatibles
v = torch.tensor([1.0, 2.0, 3.0])          # shape (3,)
m = torch.ones(4, 3)                        # shape (4, 3)
print((m + v).shape)   # v est applique a chaque ligne de m, shape (4, 3)

# indexation comme numpy
t = torch.arange(20).reshape(4, 5)
print(t)
print(t[0])          # premiere ligne
print(t[:, 1])       # deuxieme colonne
print(t[1:3, 2:4])   # sous-bloc

### Fiche technique : le broadcasting

Le broadcasting, c'est la regle qui permet a PyTorch de combiner deux tensors de shapes differentes sans que tu aies besoin de dupliquer les donnees toi-meme.

**Analogie** : une recette de cuisine prevue pour 1 personne (`shape (3,)`, un ingredient = un nombre) que tu appliques automatiquement a 4 convives (`shape (4, 3)`) : PyTorch "repete" la recette de base pour chaque convive, sans que tu aies a la recopier 4 fois.

Regle pratique (a comparer dimension par dimension, en partant de la droite) :
```
m : (4, 3)
v :    (3,)
------------
resultat : (4, 3)   # v est "etire" pour matcher les 4 lignes
```
Deux dimensions sont compatibles si elles sont egales, **ou** si l'une des deux vaut 1. Sinon -> erreur.

### Exercice 1 (5 min)
1. Cree un tensor `x` de shape (5, 3) avec des valeurs aleatoires (`torch.randn`)
2. Calcule la moyenne de chaque colonne (`.mean(dim=0)`)
3. Calcule la somme de chaque ligne (`.sum(dim=1)`)
4. Multiplie `x` par 2 et ajoute 1, sans boucle for

In [6]:
# ecris ta reponse ici
x = None  # TODO


## 2. Autograd : les gradients automatiques

C'est le coeur de PyTorch. Quand un tensor a `requires_grad=True`, PyTorch construit un graphe de calcul et peut calculer automatiquement les derivees avec `.backward()`.

C'est ce qui permet d'entrainer un reseau de neurones : on calcule une loss, on appelle `.backward()`, et PyTorch calcule le gradient de la loss par rapport a chaque parametre.

**Analogie** : imagine une chaine de production ou chaque poste de travail transforme un produit (une operation mathematique). L'autograd, c'est un contremaitre qui note, a chaque poste, "de combien le produit final change si j'augmente legerement mon reglage ici". `.backward()`, c'est remonter la chaine depuis le produit fini jusqu'au premier poste, en accumulant cette information a chaque etape c'est exactement la regle de la chaine (chain rule) du calcul differentiel.

In [ ]:
# exemple simple : y = x^2, dy/dx = 2x
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2

y.backward()          # calcule dy/dx
print("gradient dy/dx en x=3:", x.grad)   # doit etre 6.0

### Schema : le graphe de calcul de `y = x**2`

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

fig, ax = plt.subplots(figsize=(7, 3))
ax.set_xlim(0, 10)
ax.set_ylim(0, 4)
ax.axis("off")

def box(x, y, text, color="#cfe8ff"):
    rect = mpatches.FancyBboxPatch((x, y), 1.8, 1.2, boxstyle="round,pad=0.08",
                                    linewidth=1.5, edgecolor="#2b6cb0", facecolor=color)
    ax.add_patch(rect)
    ax.text(x + 0.9, y + 0.6, text, ha="center", va="center", fontsize=12)

box(0.5, 1.4, "x = 3.0\n(requires_grad)")
box(4.0, 1.4, "y = x**2\n= 9.0")

ax.annotate("", xy=(4.0, 2.0), xytext=(2.3, 2.0),
            arrowprops=dict(arrowstyle="->", lw=2, color="#2b6cb0"))
ax.text(3.15, 2.25, "forward", ha="center", fontsize=10, color="#2b6cb0")

ax.annotate("", xy=(2.3, 1.0), xytext=(4.0, 1.0),
            arrowprops=dict(arrowstyle="->", lw=2, color="#c53030"))
ax.text(3.15, 0.65, "backward: dy/dx = 2x = 6.0", ha="center", fontsize=10, color="#c53030")

plt.title("Graphe de calcul : forward (bleu) puis backward (rouge)")
plt.tight_layout()
plt.show()

In [ ]:
# exemple avec plusieurs variables : loss simple de regression
w = torch.tensor(0.5, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

x = torch.tensor(2.0)
y_vrai = torch.tensor(5.0)

y_pred = w * x + b
loss = (y_pred - y_vrai) ** 2   # erreur quadratique

loss.backward()
print("gradient par rapport a w:", w.grad)
print("gradient par rapport a b:", b.grad)

# IMPORTANT : les gradients s'accumulent, il faut les remettre a zero
# entre chaque etape d'entrainement (on le voit dans la boucle plus bas)

### Fiche technique : autograd en detail

| Element | Role | Piege frequent |
|---|---|---|
| `requires_grad=True` | active le suivi des operations sur ce tensor | seuls les tensors de type float peuvent l'avoir |
| `.grad` | contient le gradient accumule apres `.backward()`, `None` avant | s'accumule a chaque `.backward()`  il faut le remettre a zero explicitement |
| `.backward()` | remonte le graphe et calcule tous les gradients | ne fonctionne que sur un tensor **scalaire** (une seule valeur), sinon il faut passer un argument `gradient=...` |
| `tensor.detach()` | sort un tensor du graphe de calcul (copie qui ne suit plus les gradients) | utile pour "geler" une valeur ou l'utiliser hors entrainement |
| `with torch.no_grad():` | desactive completement le suivi dans ce bloc | a utiliser systematiquement pour l'evaluation/inference, ca economise memoire et calcul |
| leaf tensor | un tensor cree directement (pas issu d'une operation), comme `w` et `b` ici | seuls les leaf tensors avec `requires_grad=True` stockent un `.grad` ; un tensor intermediaire (comme `y_pred`) n'en a pas par defaut |

**Analogie pour `detach()` / `no_grad()`** : c'est debrancher le contremaitre. Tu continues a fabriquer le produit (faire les calculs), mais plus personne ne note comment chaque reglage l'a influence. Plus rapide, moins de memoire utilisee parfait quand tu evalues un modele deja entraine et que tu n'as plus besoin d'apprendre.

In [ ]:
# detach() et no_grad() en pratique
w = torch.tensor(2.0, requires_grad=True)
y = w ** 2

y_detache = y.detach()
print("y suit les gradients:", y.requires_grad)
print("y_detache suit les gradients:", y_detache.requires_grad)

with torch.no_grad():
    z = w ** 2   # ce calcul n'est PAS suivi par autograd
    print("z suit les gradients:", z.requires_grad)

### Exercice 2 (5 min)
Avec `w = torch.tensor(2.0, requires_grad=True)`, calcule `z = 3*w**3 + 2*w`, appelle `.backward()`, et verifie que `w.grad` correspond bien a la derivee `9*w**2 + 2` evaluee en w=2 (donc 38.0).

In [11]:
# ecris ta reponse ici
w = None  # TODO


## 3. Construire un modele avec nn.Module

En pratique on n'ecrit jamais les gradients a la main. On definit un modele en heritant de `nn.Module`, avec :
- `__init__` : on declare les couches (les "layers")
- `forward` : on decrit comment les donnees traversent le modele

**Analogie** : un `nn.Module`, c'est une chaine de montage. `__init__`, c'est la liste des machines qu'on installe sur la chaine (perceuse, peinture, controle qualite...). `forward`, c'est l'ordre dans lequel la piece passe par ces machines. Une meme machine (`nn.Linear`) peut d'ailleurs etre reutilisee a plusieurs endroits de la chaine.

Ici un petit MLP (multi-layer perceptron) pour de la classification binaire, mais la structure est la meme pour n'importe quel modele.

In [ ]:
class PetitMLP(nn.Module):
    def __init__(self, dim_entree, dim_cachee, dim_sortie):
        super().__init__()
        self.couche1 = nn.Linear(dim_entree, dim_cachee)
        self.activation = nn.ReLU()
        self.couche2 = nn.Linear(dim_cachee, dim_sortie)

    def forward(self, x):
        x = self.couche1(x)
        x = self.activation(x)
        x = self.couche2(x)
        return x

modele = PetitMLP(dim_entree=4, dim_cachee=16, dim_sortie=1).to(device)
print(modele)

# test rapide avec un batch de 8 exemples, 4 features chacun
batch_exemple = torch.randn(8, 4).to(device)
sortie = modele(batch_exemple)
print("shape de sortie:", sortie.shape)   # (8, 1)

### Fiche technique : `nn.Linear(in_features, out_features, bias=True)`

C'est la couche la plus fondamentale ("fully connected" / "dense"). Elle calcule :

```
sortie = entree @ W.T + b
```

| Parametre | Ce que c'est | Exemple ici |
|---|---|---|
| `in_features` | taille du vecteur d'entree (nb de features par exemple) | `4` |
| `out_features` | taille du vecteur de sortie (nb de neurones de la couche) | `16` puis `1` |
| `bias` (par defaut `True`) | ajoute un terme constant `b` appris, en plus de la partie lineaire | rarement desactive, sauf juste avant une BatchNorm |

`W` a la shape `(out_features, in_features)` et `b` a la shape `(out_features,)`. Les deux sont des `nn.Parameter` des tensors avec `requires_grad=True` automatiquement, et que `optimizer.step()` saura mettre a jour.

**Analogie** : chaque neurone de sortie est un "juge" qui regarde toutes les features d'entree, leur applique un poids de confiance different (`W`), et ajoute un biais personnel (`b`) avant de rendre son verdict.

### Fiche technique : les fonctions d'activation

Sans fonction d'activation entre les couches, empiler des `nn.Linear` reviendrait a une seule transformation lineaire geante (inutile). L'activation introduit de la non-linearite, ce qui permet au reseau d'apprendre des motifs complexes.

| Fonction | Formule | Usage typique | Analogie |
|---|---|---|---|
| `nn.ReLU()` | `max(0, x)` | couches cachees, choix par defaut | un videur de boite de nuit : laisse passer tout ce qui est positif, bloque tout le reste |
| `nn.Sigmoid()` | `1 / (1 + e^-x)` | sortie d'une classification binaire (probabilite entre 0 et 1) | un thermometre de confiance, toujours entre 0% et 100% |
| `nn.Tanh()` | similaire a Sigmoid mais entre -1 et 1 | couches cachees de certains reseaux recurrents | comme Sigmoid mais centre sur 0, utile quand on veut aussi representer une direction negative |

In [ ]:
# schema : a quoi ressemblent ces fonctions d'activation
import matplotlib.pyplot as plt
import torch

xs = torch.linspace(-5, 5, 200)
relu = torch.relu(xs)
sigmoid = torch.sigmoid(xs)
tanh = torch.tanh(xs)

plt.figure(figsize=(7, 4))
plt.plot(xs, relu, label="ReLU", linewidth=2)
plt.plot(xs, sigmoid, label="Sigmoid", linewidth=2)
plt.plot(xs, tanh, label="Tanh", linewidth=2)
plt.axhline(0, color="gray", linewidth=0.5)
plt.axvline(0, color="gray", linewidth=0.5)
plt.legend()
plt.title("Fonctions d'activation courantes")
plt.xlabel("entree")
plt.ylabel("sortie")
plt.tight_layout()
plt.show()

### Schema : l'architecture de `PetitMLP`

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.set_xlim(0, 12)
ax.set_ylim(0, 4)
ax.axis("off")

def box(x, w, text, color):
    rect = mpatches.FancyBboxPatch((x, 1.2), w, 1.6, boxstyle="round,pad=0.08",
                                    linewidth=1.5, edgecolor="#333", facecolor=color)
    ax.add_patch(rect)
    ax.text(x + w / 2, 2.0, text, ha="center", va="center", fontsize=11)

steps = [
    (0.3, 1.8, "entree\n(batch, 4)", "#e2e8f0"),
    (2.6, 2.2, "nn.Linear\n(4 -> 16)", "#cfe8ff"),
    (5.3, 1.8, "nn.ReLU()", "#fefcbf"),
    (7.6, 2.2, "nn.Linear\n(16 -> 1)", "#cfe8ff"),
    (10.3, 1.5, "sortie\n(batch, 1)", "#e2e8f0"),
]

for x, w, text, color in steps:
    box(x, w, text, color)

edges = [(2.1, 2.6), (4.8, 5.3), (7.1, 7.6), (9.8, 10.3)]
for x0, x1 in edges:
    ax.annotate("", xy=(x1, 2.0), xytext=(x0, 2.0),
                arrowprops=dict(arrowstyle="->", lw=2, color="#333"))

plt.title("PetitMLP : entree -> Linear -> ReLU -> Linear -> sortie")
plt.tight_layout()
plt.show()

## 4. Boucle d'entrainement complete

C'est le pattern a connaitre par coeur, on le reutilise partout :

1. `optimizer.zero_grad()` : remettre les gradients a zero
2. `y_pred = modele(x)` : forward pass
3. `loss = fonction_loss(y_pred, y_vrai)` : calculer l'erreur
4. `loss.backward()` : calculer les gradients
5. `optimizer.step()` : mettre a jour les poids

**Analogie generale** : c'est un eleve qui s'entraine sur des exercices. Il essaie une reponse (forward), compare a la correction et mesure son erreur (loss), comprend *pourquoi* il s'est trompe et sur quel point exactement (backward), puis ajuste sa methode en consequence (step). Il recommence l'exercice suivant en ayant oublie l'erreur precedente (zero_grad) — sinon il melangerait les corrections de plusieurs exercices differents.

On genere des donnees synthetiques pour un probleme de classification binaire et on entraine le modele dessus.

### Fiche technique : les fonctions de loss

| Fonction | Pour quel probleme | Ce qu'elle mesure | Analogie |
|---|---|---|---|
| `nn.MSELoss()` | regression (predire un nombre continu) | moyenne des `(prediction - verite)^2` | un archer note sur la distance au centre de la cible, au carre  rater de loin coute beaucoup plus cher que rater de peu |
| `nn.BCEWithLogitsLoss()` | classification binaire (oui/non) | entropie croisee binaire, appliquee directement sur les logits (avant sigmoid) | un meteorologue qui annonce "80% de chance de pluie" est severement penalise s'il ne pleut pas, et recompense s'il pleut vraiment |
| `nn.CrossEntropyLoss()` | classification multi-classe (choisir parmi N categories) | entropie croisee sur les logits, avec softmax integre | un QCM a plusieurs choix : la penalite depend de la confiance donnee a la bonne reponse |

**Piege classique** : `BCEWithLogitsLoss` et `CrossEntropyLoss` attendent des **logits** en entree (sortie brute du modele, avant activation), pas des probabilites deja passees par `sigmoid`/`softmax`  c'est fait expres, pour la stabilite numerique. Ne pas ajouter de `sigmoid()` avant `BCEWithLogitsLoss`, sinon la loss est fausse.

### Fiche technique : `optim.Adam(parametres, lr=..., betas=(0.9, 0.999), eps=1e-8, weight_decay=0)`

| Parametre | Role | Valeur par defaut | Effet si on la change |
|---|---|---|---|
| `lr` (learning rate) | taille du pas a chaque mise a jour | pas de defaut universel, souvent `1e-3` a `1e-2` | trop grand -> le modele diverge/oscille ; trop petit -> l'entrainement est tres lent |
| `betas` | vitesse d'oubli des moyennes mobiles du gradient (momentum) | `(0.9, 0.999)` | rarement modifie en debut de parcours |
| `weight_decay` | penalite qui pousse les poids vers 0 (regularisation L2) | `0` | trop eleve -> le modele sous-apprend |

**SGD vs Adam** : `optim.SGD` avance d'un pas fixe dans la direction du gradient simple, mais sensible au choix de `lr` et lent sur des paysages complexes. `optim.Adam` adapte automatiquement la taille du pas pour chaque parametre, en se souvenant des gradients recents c'est un bon choix par defaut car il demande moins de reglage manuel.

**Analogie** : `SGD`, c'est marcher vers le bas d'une colline dans le brouillard, a pas fixe, en suivant juste la pente locale. `Adam`, c'est la meme descente mais avec un baton de marche intelligent qui raccourcit automatiquement le pas dans les zones accidentees et l'allonge sur le plat.

In [ ]:
# donnees synthetiques : classification binaire simple
torch.manual_seed(0)
N = 200
X = torch.randn(N, 4)
# regle arbitraire : classe 1 si somme des features > 0
y_vrai = (X.sum(dim=1) > 0).float().unsqueeze(1)

X, y_vrai = X.to(device), y_vrai.to(device)

modele = PetitMLP(dim_entree=4, dim_cachee=16, dim_sortie=1).to(device)
fonction_loss = nn.BCEWithLogitsLoss()   # loss adaptee a la classification binaire
optimizer = optim.Adam(modele.parameters(), lr=0.01)

nb_epochs = 100
for epoch in range(nb_epochs):
    optimizer.zero_grad()
    y_pred = modele(X)
    loss = fonction_loss(y_pred, y_vrai)
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"epoch {epoch:3d} | loss {loss.item():.4f}")

print("loss finale:", loss.item())

### Schema : le cycle de la boucle d'entrainement

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.axis("off")

steps = ["1. zero_grad()", "2. forward\n(modele)", "3. loss()", "4. backward()", "5. step()"]
n = len(steps)
angles = np.linspace(np.pi / 2, np.pi / 2 - 2 * np.pi, n, endpoint=False)

positions = [(np.cos(a), np.sin(a)) for a in angles]

for (x, y), text in zip(positions, steps):
    ax.scatter([x], [y], s=3500, color="#cfe8ff", edgecolors="#2b6cb0", linewidths=1.5, zorder=2)
    ax.text(x, y, text, ha="center", va="center", fontsize=9, zorder=3)

for i in range(n):
    x0, y0 = positions[i]
    x1, y1 = positions[(i + 1) % n]
    ax.annotate("", xy=(x1 * 0.82, y1 * 0.82), xytext=(x0 * 0.82, y0 * 0.82),
                arrowprops=dict(arrowstyle="->", lw=2, color="#333",
                                 connectionstyle="arc3,rad=0.15"))

plt.title("La boucle d'entrainement : un cycle qu'on repete a chaque epoch")
plt.tight_layout()
plt.show()

In [ ]:
# evaluer la precision du modele entraine
with torch.no_grad():
    y_pred = modele(X)
    y_pred_classe = (torch.sigmoid(y_pred) > 0.5).float()
    precision = (y_pred_classe == y_vrai).float().mean()
    print("precision sur les donnees d'entrainement:", precision.item())

### Points a retenir (cheatsheet rapide)

- `requires_grad=True` -> PyTorch suit les operations pour calculer les gradients
- `optimizer.zero_grad()` avant chaque `backward()`, sinon les gradients s'accumulent
- `with torch.no_grad():` quand on evalue, pour ne pas gaspiller de memoire sur les gradients
- `nn.Linear(in, out)` = une couche entierement connectee, calcule `x @ W.T + b`
- Activation : `nn.ReLU()` en couche cachee par defaut, `nn.Sigmoid()`/softmax seulement en sortie si besoin d'une probabilite
- Choix de loss : `nn.MSELoss()` pour la regression, `nn.BCEWithLogitsLoss()` pour classification binaire, `nn.CrossEntropyLoss()` pour classification multi-classe
- `optim.Adam(modele.parameters(), lr=...)` est un bon choix par defaut
- `.item()` pour recuperer un scalaire Python a partir d'un tensor a un seul element (pratique pour les print)

## 5. Exercice final chrono (15-20 min)

A faire seul, sans aide, comme en condition d'epreuve.

Consignes :
1. Genere des donnees synthetiques pour une regression : `X` de shape (100, 1), et `y = 3*X + 2 + bruit` (ajoute un peu de bruit gaussien avec `torch.randn`)
2. Definis un modele avec une seule couche `nn.Linear(1, 1)`
3. Entraine-le avec `nn.MSELoss()` et `optim.SGD` ou `optim.Adam`, pendant 200 epochs
4. Affiche la loss toutes les 50 epochs
5. Bonus : affiche les valeurs apprises de poids et biais (`modele.weight`, `modele.bias`), ca doit se rapprocher de 3 et 2

In [18]:
# ecris ta solution ici





## 6. Pour aller plus loin

Cette section est a travailler en autonomie pendant la semaine, pas pendant la soiree. Chaque point peut se traiter independamment.

### 6.1 Dataset et DataLoader
En vrai, on ne passe presque jamais toutes les donnees d'un coup comme dans l'exemple ci-dessus (`X` entier). On decoupe en petits lots ("mini-batches") avec `torch.utils.data.Dataset` et `DataLoader`. Analogie : au lieu de corriger toutes les copies d'une classe d'un coup, le prof corrige par paquets de 10, et fait une pause (mise a jour des poids) entre chaque paquet.

```python
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(X, y_vrai)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

for batch_X, batch_y in loader:
    ...  # meme boucle d'entrainement, mais sur un batch a la fois
```

### 6.2 Sauvegarder et recharger un modele
```python
torch.save(modele.state_dict(), "modele.pt")

modele2 = PetitMLP(4, 16, 1)
modele2.load_state_dict(torch.load("modele.pt"))
modele2.eval()  # bascule en mode evaluation (desactive dropout/batchnorm)
```
`.state_dict()` est un dictionnaire `{nom_du_parametre: tensor}` c'est ca qu'on sauvegarde, pas le code du modele lui-meme.

### 6.3 `.train()` vs `.eval()`
Certaines couches (Dropout, BatchNorm) se comportent differemment a l'entrainement et a l'evaluation. `modele.train()` et `modele.eval()` bascule ce mode. Reflexe a prendre : `.eval()` avant toute inference/evaluation, `.train()` avant de reprendre l'entrainement.

### 6.4 Pieges frequents (a debugger volontairement)
Pour bien comprendre, essaie de provoquer ces erreurs volontairement puis de les corriger :
- Oublier `optimizer.zero_grad()` pendant 20 epochs : que se passe-t-il sur la courbe de loss ?
- Utiliser une `lr` beaucoup trop grande (`lr=10`) : que se passe-t-il ?
- Melanger un tensor sur CPU et un tensor sur GPU dans la meme operation
- Appeler `.backward()` deux fois de suite sans refaire de forward pass

### 6.5 Pistes pour 
- S'entrainer a lire rapidement une architecture de modele (compter les parametres, deviner la shape de sortie de chaque couche a la main)
- Pratiquer le debug de code PyTorch fourni avec des bugs volontaires (erreurs de shape, mauvaise loss, gradient qui explose)
- Revoir les bases de calcul differentiel (regle de la chaine) puisque l'autograd repose entierement dessus
- S'exercer a implementer rapidement une boucle d'entrainement complete de memoire, sans regarder ce notebook